In [1]:
import pandas as pd
import numpy as np
import pmdarima as pm
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error, mean_absolute_error

In [2]:
print("Loading data for statistical baseline...")
df = pd.read_csv('data/bike_sharing.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()

Loading data for statistical baseline...


In [3]:
# Resample to daily for simpler ARIMA/SARIMA to speed things up
daily_demand = df['demand'].resample('D').sum()

In [4]:
train_size = int(len(daily_demand) * 0.8)
train, test = daily_demand.iloc[:train_size], daily_demand.iloc[train_size:]

In [5]:
print("Fitting ARIMA...")
arima_model = pm.auto_arima(train, seasonal=False, trace=True, suppress_warnings=True)
arima_preds = arima_model.predict(n_periods=len(test))

Fitting ARIMA...
Performing stepwise search to minimize aic


 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.28 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=3220.869, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=3156.678, Time=0.07 sec


 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.19 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=3219.900, Time=0.01 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=3137.395, Time=0.10 sec
 ARIMA(3,1,0)(0,0,0)[0] intercept   : AIC=3124.508, Time=0.05 sec


 ARIMA(4,1,0)(0,0,0)[0] intercept   : AIC=3111.377, Time=0.16 sec
 ARIMA(5,1,0)(0,0,0)[0] intercept   : AIC=3106.723, Time=0.19 sec


 ARIMA(5,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.60 sec


 ARIMA(4,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.57 sec
 ARIMA(5,1,0)(0,0,0)[0]             : AIC=3117.890, Time=0.11 sec

Best model:  ARIMA(5,1,0)(0,0,0)[0] intercept
Total fit time: 2.393 seconds


In [6]:
arima_mape = mean_absolute_percentage_error(test, arima_preds)
arima_rmse = root_mean_squared_error(test, arima_preds)
arima_mae = mean_absolute_error(test, arima_preds)

In [7]:
print(f"ARIMA -> MAPE: {arima_mape:.4f}, RMSE: {arima_rmse:.4f}, MAE: {arima_mae:.4f}")

ARIMA -> MAPE: 0.0117, RMSE: 49.1432, MAE: 40.2764


In [8]:
print("Fitting SARIMA (weekly seasonality)...")
# Using weekly seasonality (m=7)
sarima_model = pm.auto_arima(train, seasonal=True, m=7, trace=True, suppress_warnings=True)
sarima_preds = sarima_model.predict(n_periods=len(test))

Fitting SARIMA (weekly seasonality)...
Performing stepwise search to minimize aic


 ARIMA(2,1,2)(1,0,1)[7] intercept   : AIC=inf, Time=0.90 sec
 ARIMA(0,1,0)(0,0,0)[7] intercept   : AIC=3220.869, Time=0.01 sec


 ARIMA(1,1,0)(1,0,0)[7] intercept   : AIC=3153.524, Time=0.23 sec


 ARIMA(0,1,1)(0,0,1)[7] intercept   : AIC=inf, Time=0.48 sec
 ARIMA(0,1,0)(0,0,0)[7]             : AIC=3219.900, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[7] intercept   : AIC=3156.678, Time=0.06 sec


 ARIMA(1,1,0)(2,0,0)[7] intercept   : AIC=3154.906, Time=0.40 sec


...

 ARIMA(5,1,0)(1,0,0)[7] intercept   : AIC=3104.624, Time=0.62 sec


 ARIMA(5,1,0)(1,0,2)[7] intercept   : AIC=3107.552, Time=1.30 sec


 ARIMA(5,1,1)(0,0,1)[7] intercept   : AIC=inf, Time=1.05 sec


 ARIMA(4,1,1)(0,0,1)[7] intercept   : AIC=inf, Time=0.88 sec


 ARIMA(5,1,0)(0,0,1)[7]             : AIC=3118.101, Time=0.22 sec

Best model:  ARIMA(5,1,0)(0,0,1)[7] intercept
Total fit time: 17.333 seconds


In [9]:
sarima_mape = mean_absolute_percentage_error(test, sarima_preds)
sarima_rmse = root_mean_squared_error(test, sarima_preds)
sarima_mae = mean_absolute_error(test, sarima_preds)

In [10]:
print(f"SARIMA -> MAPE: {sarima_mape:.4f}, RMSE: {sarima_rmse:.4f}, MAE: {sarima_mae:.4f}")

SARIMA -> MAPE: 0.0110, RMSE: 46.3584, MAE: 37.8608
